In [95]:
import pandas as pd
from sklearn.ensemble import StackingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [96]:
train_data_path = "./home-data-for-ml-course/train.csv"

read_train_data = pd.read_csv(train_data_path)

read_train_data.columns
read_train_data.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [97]:
# Prediction Data 

y = read_train_data.SalePrice 

house_features = [
    'OverallQual',  # paling penting!
    'GrLivArea',    # luas ruang hidup
    'TotalBsmtSF',  # luas basement
    'GarageCars',   # kapasitas garasi
    'YearBuilt',    # tahun dibangun
    'FullBath',     # kamar mandi
    'TotRmsAbvGrd', # total ruangan
    'LotArea',      # luas tanah
    '1stFlrSF',     # lantai 1
    'GarageArea',   # luas garasi
]


X = read_train_data[house_features].select_dtypes(include=['int64', 'float64'])
print(X)
X.isnull().sum()


      OverallQual  GrLivArea  TotalBsmtSF  GarageCars  YearBuilt  FullBath  \
0               7       1710          856           2       2003         2   
1               6       1262         1262           2       1976         2   
2               7       1786          920           2       2001         2   
3               7       1717          756           3       1915         1   
4               8       2198         1145           3       2000         2   
...           ...        ...          ...         ...        ...       ...   
1455            6       1647          953           2       1999         2   
1456            6       2073         1542           2       1978         2   
1457            7       2340         1152           1       1941         2   
1458            5       1078         1078           1       1950         1   
1459            5       1256         1256           1       1965         1   

      TotRmsAbvGrd  LotArea  1stFlrSF  GarageArea  
0          

OverallQual     0
GrLivArea       0
TotalBsmtSF     0
GarageCars      0
YearBuilt       0
FullBath        0
TotRmsAbvGrd    0
LotArea         0
1stFlrSF        0
GarageArea      0
dtype: int64

In [98]:
# Max_leaf_Node wiht DecisionTreeRegressor
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 1)


In [99]:
# Train Model & MAE (Train)

model_leaf = DecisionTreeRegressor(max_leaf_nodes=100, random_state=1)

model_leaf.fit(train_X, train_y)

val_leaf_predictions = model_leaf.predict(val_X)
val_leaf_mae = mean_absolute_error(val_y, val_leaf_predictions)

print("Validation MAE for best value of max_leaf_nodes: {:,.0f}".format(val_leaf_mae))

estimators = [('hgb', HistGradientBoostingRegressor(max_iter=300, random_state = 1)),(
    'rf', RandomForestRegressor(n_estimators=300, random_state = 1),
    
)]

train_model = StackingRegressor(estimators=estimators, final_estimator=LinearRegression())

train_model.fit(train_X, train_y)

# MAE checking 
val_predictions = train_model.predict(val_X)

mae = mean_absolute_error(val_y, val_predictions)
print("Validation MAE for best value of max_leaf_nodes: {:,.0f}".format(mae))


Validation MAE for best value of max_leaf_nodes: 23,590
Validation MAE for best value of max_leaf_nodes: 19,441


In [100]:
# Test prediction 

test_data = "./home-data-for-ml-course/test.csv"

read_test_data = pd.read_csv(test_data)

read_test_data.isnull().sum()

filtered_read_data = read_test_data.select_dtypes(include=["int64", "float64"])

test_predictions = train_model.predict(filtered_read_data[house_features])

output = pd.DataFrame({
    'Id' : read_test_data.Id,
    'SalePrice' : test_predictions
})

output.to_csv('submission.csv', index=False)